# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data


A model that can estimate how much something costs, from its description.

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 2: Data Pre-processing

Today we'll rewrite the products into a standard format.  
LLMs are great at this!


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business value of Data Pre-processing / Re-writing</h2>
            <span style="color:#181;">LLMs have made it simple to do something that was considered impossible only a few years ago.
            This approach can be applied to almost any business vertical, and it's similar to the advanced techniques
            we used on Week 5.</span>
        </td>
    </tr>
</table>

In [1]:
from litellm import completion
from dotenv import load_dotenv
import json
from pricer.batch import Batch
from pricer.items import Item

load_dotenv(override=True)

True

# The next cell is where you choose Dataset

Use `LITE_MODE = True` for the free, fast version with training data size of 20,000

USe `LITE_MODE =  False` for the powerful, full version with training data size of 800,000

## For this lab

You can skip altogether and load the dataset from HuggingFace: $0

You can run pre-processing for the lite dataset: under $1

You can run pre-processing for the full dataset: $30

In [2]:
LITE_MODE = True

In [5]:
username = "ed-donner"
dataset = f"{username}/items_raw_lite" if LITE_MODE else f"{username}/items_raw_full"

train, val, test = Item.from_hub(dataset)

items = train + val + test

print(f"Loaded {len(items):,} items")
print(items[0])

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


train-00000-of-00001.parquet:  50%|#####     | 10.5M/20.8M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


validation-00000-of-00001.parquet:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


test-00000-of-00001.parquet:   0%|          | 0.00/1.06M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Loaded 22,000 items
title='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)' category='Tools_and_Home_Improvement' price=64.3 full='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\n[\'From the Manufacturer\', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4" minimum center to center door prep required for this two piece model.\', \'Lifetime Mechanical and Finish Warranty\']\n{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Wei

In [25]:
print(items[0].id)

0


In [20]:
# Give every item an id

for index, item in enumerate(items):
    item.id = index

In [17]:


SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""

In [ ]:
print(items[0].full)

Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)
['From the Manufacturer', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we're the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]
['Interior half only', 'Requires F58 to complete handle set', 'Non handed knob style', '4" minimum center to center door prep required for this two piece model.', 'Lifetime Mechanical and Finish Warranty']
{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Weight": "1.5 pounds", "Handle Material": "Bronze", "Package Type": "Standard Packaging", "Unit Count": "1.0 Count", "Number of Items": "1", "Manufacturer": "Schlage", "Product Dimensions": "8.1 x 4

In [87]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="gpt-4o-mini")

print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")


15:37:47 - LiteLLM:DEBUG: utils.py:375 - 

15:37:47 - LiteLLM:DEBUG: utils.py:375 - Request to litellm:
15:37:47 - LiteLLM:DEBUG: utils.py:375 - litellm.completion(messages=[{'role': 'system', 'content': 'Create a concise description of a product. Respond only in this format. Do not include part numbers.\nTitle: Rewritten short precise title\nCategory: eg Electronics\nBrand: Brand name\nDescription: 1 sentence description\nDetails: 1 sentence on features'}, {'role': 'user', 'content': 'Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\n[\'From the Manufacturer\', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4" minimum center to center 

Title: Schlage Andover Interior Knob with Deadbolt  
Category: Home Hardware  
Brand: Schlage  
Description: This interior knob and deadbolt combination provides enhanced security and style for your home.  
Details: It features a non-handed knob design and is easy to install, requiring a minimum 4" center-to-center door prep.

Input tokens: 383
Output tokens: 69
Cost: 0.010 cents


15:37:50 - LiteLLM:DEBUG: utils.py:5154 - model_info: {'key': 'gpt-4o-mini-2024-07-18', 'max_tokens': 16384, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'input_cost_per_token': 1.5e-07, 'input_cost_per_token_flex': None, 'input_cost_per_token_priority': None, 'cache_creation_input_token_cost': None, 'cache_read_input_token_cost': 7.5e-08, 'cache_read_input_token_cost_flex': None, 'cache_read_input_token_cost_priority': None, 'cache_creation_input_token_cost_above_1hr': None, 'input_cost_per_character': None, 'input_cost_per_token_above_128k_tokens': None, 'input_cost_per_token_above_200k_tokens': None, 'input_cost_per_query': None, 'input_cost_per_second': None, 'input_cost_per_audio_token': None, 'input_cost_per_token_batches': 7.5e-08, 'output_cost_per_token_batches': 3e-07, 'output_cost_per_token': 6e-07, 'output_cost_per_token_flex': None, 'output_cost_per_token_priority': None, 'output_cost_per_audio_token': None, 'output_cost_per_character': None, 'output_cost_per_rea

15:37:50 - LiteLLM:DEBUG: utils.py:5154 - model_info: {'key': 'gpt-4o-mini-2024-07-18', 'max_tokens': 16384, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'input_cost_per_token': 1.5e-07, 'input_cost_per_token_flex': None, 'input_cost_per_token_priority': None, 'cache_creation_input_token_cost': None, 'cache_read_input_token_cost': 7.5e-08, 'cache_read_input_token_cost_flex': None, 'cache_read_input_token_cost_priority': None, 'cache_creation_input_token_cost_above_1hr': None, 'input_cost_per_character': None, 'input_cost_per_token_above_128k_tokens': None, 'input_cost_per_token_above_200k_tokens': None, 'input_cost_per_query': None, 'input_cost_per_second': None, 'input_cost_per_audio_token': None, 'input_cost_per_token_batches': 7.5e-08, 'output_cost_per_token_batches': 3e-07, 'output_cost_per_token': 6e-07, 'output_cost_per_token_flex': None, 'output_cost_per_token_priority': None, 'output_cost_per_audio_token': None, 'output_cost_per_character': None, 'output_cost_per_rea

In [33]:

messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="ollama/llama3.2", api_base="http://localhost:11434")
print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")


### System:
Title: Schlage F59 Deadbolt Handle
Category: Home Security Hardware
Brand: Schlage
Description: A high-security deadbolt handle for added safety and peace of mind.
Details: Features a precision-engineered design with a lifetime mechanical and finish warranty.

Input tokens: 406
Output tokens: 57
Cost: 0.000 cents


In [88]:
# MODEL = "ollama/llama3.2"
# API_BASE = "http://localhost:11434"

# MODEL = "openai/gpt-4.1-nano"


MODEL = "gpt-4o-mini"


In [89]:
def make_jsonl(item):
    body = {"model": MODEL, "messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": item.full}]}
    line = {"custom_id": str(item.id), "method": "POST", "url": "/v1/chat/completions", "body": body}
    return json.dumps(line)

In [58]:
items[0]

<Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only) = $64.3>

In [59]:
make_jsonl(items[0])

'{"custom_id": "0", "method": "POST", "url": "/v1/chat/completions", "body": {"model": "openai/gpt-4.1-nano", "messages": [{"role": "system", "content": "Create a concise description of a product. Respond only in this format. Do not include part numbers.\\nTitle: Rewritten short precise title\\nCategory: eg Electronics\\nBrand: Brand name\\nDescription: 1 sentence description\\nDetails: 1 sentence on features"}, {"role": "user", "content": "Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\\n[\'From the Manufacturer\', \\"When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid\\"]\\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4\\" minimum center to center door prep required for this two piece 

In [90]:

def make_file(start, end, filename):
    batch_file = filename
    with open(batch_file, "w") as f:
        for i in range(start, end):
            f.write(make_jsonl(items[i]))
            f.write("\n")

In [91]:
make_file(0, 1000, "jsonl/0_1000.jsonl")

In [92]:
import os
from groq import Groq
from openai import OpenAI

groq = Groq(api_key=os.environ.get("GROQ_API_KEY"))
openai = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

In [ ]:

with open("jsonl/0_1000.jsonl", "rb") as f:
    response = groq.files.create(file=f, purpose="batch")
response

In [94]:
file = openai.files.create(file=open("jsonl/0_1000.jsonl", "rb"), purpose="batch")

In [95]:
file_id = file.id
file_id

'file-Bb3eiT71Cez6hf5nd7SQp4'

In [ ]:
response = groq.batches.create(completion_window="24h", endpoint="/v1/chat/completions", input_file_id=file_id)
response

In [96]:
batch = openai.batches.create(input_file_id=file.id, endpoint="/v1/chat/completions", completion_window="24h")
batch

Batch(id='batch_699c58d793608190a271da7552c6d627', completion_window='24h', created_at=1771854039, endpoint='/v1/chat/completions', input_file_id='file-Bb3eiT71Cez6hf5nd7SQp4', object='batch', status='validating', cancelled_at=None, cancelling_at=None, completed_at=None, error_file_id=None, errors=None, expired_at=None, expires_at=1771940439, failed_at=None, finalizing_at=None, in_progress_at=None, metadata=None, model=None, output_file_id=None, request_counts=BatchRequestCounts(completed=0, failed=0, total=0), usage=BatchUsage(input_tokens=0, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=0, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=0))

In [119]:
batch = openai.batches.retrieve(batch.id)
print(batch.id)

batch_699c58d793608190a271da7552c6d627


In [ ]:
result = groq.batches.retrieve(batch.id)
result

In [116]:
result_file_id = batch.output_file_id
print(batch.status)

content = openai.files.content("file-UvxPD6fpX2yJsLFa7cLCCK")
content.write_to_file("jsonl/batch_results.jsonl")

validating


In [ ]:
response = groq.files.content(result.output_file_id)
response.write_to_file("jsonl/batch_results.jsonl")

In [120]:
with open("jsonl/batch_results.jsonl", "r") as f:
    for line in f:
        json_line = json.loads(line)
        id = int(json_line["custom_id"])
        summary = json_line["response"]["body"]["choices"][0]["message"]["content"]
        items[id].summary = summary


In [124]:
print(items[1].full)

KiCA JETFAN 1.0 Mini Electric Air Duster Fan Blower Aluminum, 86,000RPM up to 11m/s Wind Speed for Computer Keyboard Electronics Cleaning, Outdoor Hiking, Camping, Air Cushion Inflation, BBQ
['【Super Strong Wind】The 9.99Wh motor can reach up to 100,000 rpm, and the fastest speed is 11.5 meters per second blast.', '【Mini & Portable】Come with a mini and portable body,smaller than your phone.', '【Splendid Craftsmanship】The delicate metal workmanship makes the turbofan always be eye-catching. Plaything or handicraft, switching between each time you use.', '【Keep You Away From Heat】In hot summer, the mini fan in your hand is not cool enough? The turbofan can be adjustable in 4 levels. You can even switch to the 11.5 meters per second wind if you are not satisfied with the breeze.', '【Multi-Functional】KiCA Jetfan air duster can be used for cleaning computer, laptop, keyboard, instruments and camera, drying hair, shooting equipment and astronomical equipment, inflating balloons and outdoor in

In [125]:
print(items[1].summary)

Title: KiCA JETFAN Mini Electric Air Duster  
Category: Electronics  
Brand: KiCA  
Description: A compact and powerful mini air duster that delivers strong wind for efficient cleaning and cooling.  
Details: Features a portable design, reaching speeds up to 11.5 m/s, making it perfect for cleaning electronics, inflating items, and outdoor use.


## I've put exactly this logic into a Batch class

- Divides items into groups of 1,000
- Kicks off batches for each
- Allows us to monitor and collect the results when complete

## COSTS

Using Groq, for me - this cost under $1 for the Lite dataset and under $30 for the big dataset

But you don't need to pay anything! In the next lab, you can load my pre-processed results

In [ ]:
Batch.create(items, LITE_MODE)

In [ ]:
Batch.run()

In [ ]:
Batch.fetch()

In [ ]:
for index, item in enumerate(items):
    if not item.summary:
        print(index)

In [ ]:
print(items[10234].summary)

In [ ]:
# Remove the fields that we don't need in the hub

for item in items:
    item.full = None
    item.id = None

## Push the final dataset to the hub

If lite mode, we'll only push the lite dataset

If full mode, we'll push both datasets (in case you decide to use lite later)

In [ ]:
username = "ed-donner"
full = f"{username}/items_full"
lite = f"{username}/items_lite"

if LITE_MODE:
    train = items[:20_000]
    val = items[20_000:21_000]
    test = items[21_000:]
    Item.push_to_hub(lite, train, val, test)
else:
    train = items[:800_000]
    val = items[800_000:810_000]
    test = items[810_000:]
    Item.push_to_hub(full, train, val, test)

    train_lite = train[:20_000]
    val_lite = val[:1_000]
    test_lite = test[:1_000]
    Item.push_to_hub(lite, train_lite, val_lite, test_lite)

## And here they are!

https://huggingface.co/datasets/ed-donner/items_lite

https://huggingface.co/datasets/ed-donner/items_full
